# Train Scorer from MongoDB
Uses Words + UserWordProgress collections to train the scorer and save `checkpoints/scorer.pt`. Requires `MONGODB_URI`.


In [ ]:
# 1. Imports and config
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pymongo import MongoClient
from dotenv import load_dotenv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Save checkpoint in parent directory (ml_agent/checkpoints)
CKPT_DIR = Path("..") / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True, parents=True)
CKPT_PATH = CKPT_DIR / "scorer.pt"

print({"device": str(DEVICE), "ckpt": str(CKPT_PATH)})

In [ ]:
# 2. Load environment
load_dotenv()
mongo_uri = os.getenv("MONGO_URI")
if not mongo_uri:
    raise RuntimeError("MONGO_URI is required. Set it in .env or environment.")
print("Using MONGO_URI", mongo_uri)


In [ ]:
# 3. Data loading helpers
client = MongoClient(mongo_uri)
# Extract database name from URI or use default
try:
    db = client.get_default_database()
except:
    db = client["test"]  # fallback to test database (as shown in MongoDB Atlas)

print(f"Using database: {db.name}")
print(f"Available collections: {db.list_collection_names()}")

words = list(db["words"].find({}))
progress = list(db["userwordprogresses"].find({}))
print(f"Loaded {len(words)} words, {len(progress)} progress docs")

# Basic sanity
if len(words) == 0:
    print("WARNING: No words found in 'words' collection")
    print("Check if collection name is correct or if database has data")


In [ ]:
# 4. Build dataset from individual user data
print("Building dataset from individual users...")

# Get all users with expertise levels
users_list = list(db["users"].find(
    {"expertise_lvl": {"$exists": True, "$ne": None}},
    {"_id": 1, "expertise_lvl": 1}
))

if not users_list:
    print("⚠ No users found. Using fallback aggregated data.")
    # Fallback to aggregated computation
    total_attempts = sum(int(p.get("attempts", 0)) for p in progress)
    total_correct = sum(int(p.get("correct", 0)) for p in progress)
    recent_acc = (total_correct / total_attempts) if total_attempts > 0 else 0.6
    avg_mastery = (
        sum(int(p.get("mastery", 0)) for p in progress) / len(progress)
        if progress
        else 0
    )
    user_level = max(1, min(5, round(avg_mastery / 20))) if avg_mastery > 0 else 2
    user_data = [(user_level, recent_acc)]
    print({"recent_acc": recent_acc, "avg_mastery": avg_mastery, "user_level": user_level})
else:
    print(f"✓ Found {len(users_list)} users with expertise levels")
    user_data = []
    
    for user in users_list:
        from bson import ObjectId
        user_id = user["_id"]
        user_level = user["expertise_lvl"]
        
        # Get user's recent accuracy from results
        user_results = list(db["results"].find(
            {"userID": user_id},
            {"isCorrect": 1}
        ).sort("createdDate", -1).limit(20))
        
        if user_results:
            recent_acc = sum(1 for r in user_results if r.get("isCorrect", False)) / len(user_results)
        else:
            recent_acc = 0.5
        
        user_data.append((user_level, recent_acc))
    
    print(f"✓ Prepared {len(user_data)} user profiles")
    levels_dist = {}
    for lvl, _ in user_data:
        levels_dist[lvl] = levels_dist.get(lvl, 0) + 1
    print(f"  Level distribution: {dict(sorted(levels_dist.items()))}")


In [ ]:
# 5. Build dataset from all users
from typing import List

def build_state(user_level: float, difficulty: float, recent_acc: float) -> np.ndarray:
    gap = difficulty - user_level
    user_level_norm = user_level / 5.0
    gap_norm = np.tanh(gap / 3.0)
    return np.array([user_level_norm, gap_norm, recent_acc], dtype=np.float32)

X_list: List[np.ndarray] = []
y_list: List[float] = []

# Generate training samples for each user-word combination
for user_level, recent_acc in user_data:
    for w in words:
        difficulty = float(w.get("expertise_lvl", 3))
        state = build_state(user_level, difficulty, recent_acc)
        gap = abs(difficulty - user_level)
        # Reward: higher for appropriate difficulty, scaled by accuracy
        reward = (1.0 - (gap / 5.0)) * recent_acc
        X_list.append(state)
        y_list.append(reward)

X = torch.tensor(np.stack(X_list), dtype=torch.float32, device=DEVICE)
y = torch.tensor(np.array(y_list), dtype=torch.float32, device=DEVICE).unsqueeze(-1)
print(f"✓ Feature tensor {X.shape}, Target {y.shape}")
print(f"  Generated {len(X_list)} training samples from {len(user_data)} users × {len(words)} words")


In [ ]:
# 6. Define model, train, and save to MongoDB
import io
from datetime import datetime

class Scorer(nn.Module):
    def __init__(self, state_dim: int = 3, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = Scorer().to(DEVICE)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

epochs = 50
final_loss = 0.0
for epoch in range(epochs):
    opt.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    final_loss = float(loss.item())
    if (epoch + 1) % 10 == 0:
        print({"epoch": epoch + 1, "loss": final_loss})

# Save to local file
torch.save(model.state_dict(), CKPT_PATH)
print(f"✓ Saved to local file: {CKPT_PATH}")

# Save to MongoDB
def save_scorer_to_mongo():
    """Save Scorer model to MongoDB ml_models collection"""
    # Serialize model to bytes
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    buffer.seek(0)
    model_bytes = buffer.read()
    
    model_doc = {
        "model_name": "vocab_scorer",
        "model_type": "pytorch_scorer",
        "model_data": model_bytes,
        "metadata": {
            "state_dim": 3,
            "hidden_dim": 64,
            "epochs": epochs,
            "final_loss": final_loss,
            "training_samples": len(X),
            "num_users": len(user_data) if 'user_data' in locals() else 1,
            "num_words": len(words),
            "training_date": datetime.utcnow(),
        },
        "created_at": datetime.utcnow(),
        "updated_at": datetime.utcnow()
    }
    
    # Upsert (update if exists, insert if new)
    result = db["ml_models"].update_one(
        {"model_name": "vocab_scorer"},
        {"$set": model_doc},
        upsert=True
    )
    
    if result.upserted_id:
        print(f"✓ Saved new Scorer model to MongoDB (ID: {result.upserted_id})")
    else:
        print(f"✓ Updated existing Scorer model in MongoDB")

save_scorer_to_mongo()

# Test loading from MongoDB
def load_scorer_from_mongo():
    """Load Scorer model from MongoDB"""
    model_doc = db["ml_models"].find_one({"model_name": "vocab_scorer"})
    
    if not model_doc:
        print("⚠ No Scorer model found in MongoDB")
        return None
    
    # Deserialize model from bytes
    buffer = io.BytesIO(model_doc["model_data"])
    state_dict = torch.load(buffer, map_location=DEVICE)
    
    test_model = Scorer().to(DEVICE)
    test_model.load_state_dict(state_dict)
    
    print(f"✓ Loaded Scorer model from MongoDB")
    print(f"  Trained on: {model_doc['metadata'].get('training_date', 'Unknown')}")
    print(f"  Final loss: {model_doc['metadata'].get('final_loss', 'Unknown'):.6f}")
    return test_model

print("\nTesting model loading:")
loaded_model = load_scorer_from_mongo()


## Notes
- Ensure `MONGODB_URI` is set in environment or `.env`.
- Checkpoint saved to `ml_agent/checkpoints/scorer.pt` and consumed by `service.py`.
